# Laboratorio 03. MLP con NumPy

**Pregunta central:** ¿Puede una capa oculta con activación no lineal aprender la tabla XOR?

## Objetivos

- Definir formas de entrada, pesos y activaciones.
- Ejecutar forward, pérdida, gradientes manuales y actualización con NumPy.
- Visualizar la frontera aprendida y discutir el alcance del ejemplo.

## Prerrequisitos y conexión

Extiende la neurona y el perceptrón del laboratorio 02. Se requieren productos matriciales, regla de la cadena y álgebra con arreglos.

## Teoría breve

En una red de dos capas, $H=\tanh(XW_1+b_1)$ y $\hat{y}=\sigma(HW_2+b_2)$. La capa oculta transforma la representación para que XOR pueda separarse.

## Problema y datos

Usamos las cuatro combinaciones binarias de XOR. La tabla es exhaustiva para esta demostración y se construye localmente.

## Evaluación

Seguimos BCE por época, exactitud en las cuatro entradas y mapa de probabilidad. La exactitud verifica representación sobre esta tabla; no estima desempeño en una población.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 2026
rng = np.random.default_rng(SEED)
plt.rcParams['figure.figsize'] = (7, 4)

**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Propagar con NumPy
Calculamos pesos, sesgos, activación ReLU y dimensiones intermedias.


In [ ]:
X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]], dtype=np.float32)
W1 = np.array([[1, -1], [-1, 1]], dtype=np.float32); b1 = np.array([0, 0], dtype=np.float32)
W2 = np.array([[1], [1]], dtype=np.float32); b2 = np.array([-0.5], dtype=np.float32)
z1 = X @ W1 + b1
h = np.maximum(z1, 0)
output = h @ W2 + b2
print('X:', X.shape, 'z1:', z1.shape, 'h:', h.shape, 'output:', output.shape)
print(output.ravel())


**Resultado e interpretación.** Cada matriz debe tener dimensiones compatibles: las unidades ocultas forman una representación intermedia; sus valores dependen de los parámetros iniciales.


## Conexión con el siguiente laboratorio

El siguiente laboratorio implementa el mismo tipo de propagación hacia adelante con tensores y capas de PyTorch.


### Entrenar XOR con el ciclo NumPy visible

Usamos las cuatro combinaciones posibles como entrenamiento completo. No existe una muestra de prueba independiente para esta tabla exhaustiva; el objetivo es verificar el mecanismo y no estimar generalización a una población.

In [ ]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y = np.array([[0], [1], [1], [0]], dtype=np.float32)
hidden_units = 4
W1 = rng.normal(0, 0.6, size=(2, hidden_units)).astype(np.float32)
b1 = np.zeros((1, hidden_units), dtype=np.float32)
W2 = rng.normal(0, 0.6, size=(hidden_units, 1)).astype(np.float32)
b2 = np.zeros((1, 1), dtype=np.float32)
learning_rate = 0.5
loss_history = []
for epoch in range(3000):
    z1 = X @ W1 + b1
    hidden = np.tanh(z1)
    logits = hidden @ W2 + b2
    probabilities = 1 / (1 + np.exp(-logits))
    loss = np.mean(np.logaddexp(0, logits) - y * logits)
    d_logits = (probabilities - y) / len(X)
    dW2 = hidden.T @ d_logits
    db2 = d_logits.sum(axis=0, keepdims=True)
    d_hidden = d_logits @ W2.T
    d_z1 = d_hidden * (1 - hidden**2)
    dW1 = X.T @ d_z1
    db1 = d_z1.sum(axis=0, keepdims=True)
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    loss_history.append(float(loss))
predictions = (probabilities >= 0.5).astype(int)
print('BCE inicial/final:', round(loss_history[0], 4), round(loss_history[-1], 4))
print('probabilidades:', probabilities.ravel().round(3))
print('predicciones XOR:', predictions.ravel(), '| exactitud:', (predictions == y).mean())
xx, yy = np.meshgrid(np.linspace(-0.2, 1.2, 100), np.linspace(-0.2, 1.2, 100))
grid = np.column_stack([xx.ravel(), yy.ravel()])
grid_hidden = np.tanh(grid @ W1 + b1)
grid_probability = 1 / (1 + np.exp(-(grid_hidden @ W2 + b2)))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(loss_history)
axes[0].set(xlabel='Época', ylabel='BCE', title='Entrenamiento NumPy')
axes[1].contourf(xx, yy, grid_probability.reshape(xx.shape), levels=20, cmap='coolwarm', alpha=0.8)
axes[1].scatter(X[:, 0], X[:, 1], c=y.ravel(), edgecolor='black', cmap='coolwarm', s=80)
axes[1].set(xlabel='x1', ylabel='x2', title='Probabilidad de clase 1')
plt.tight_layout()
plt.show()

## Limitaciones y conclusiones

La tabla XOR contiene solo cuatro entradas y no es un conjunto de prueba independiente: verificamos una capacidad de representación, no generalización estadística. El gradiente implementado muestra las dimensiones y la regla de la cadena que luego automatizará PyTorch.

## Ejercicios

1. **Guiado:** cambia `hidden_units` de 4 a 2, reinicia la semilla y registra si las cuatro etiquetas siguen siendo correctas.
2. **Independiente:** reemplaza `tanh` por ReLU y deriva su gradiente por tramos para actualizar la capa oculta.
3. **Desafío:** comprueba mediante diferencias finitas el gradiente de un elemento de `W1` y fija una tolerancia justificada.

## Conexión con el siguiente laboratorio

En el siguiente notebook replicarás el forward con capas PyTorch y contrastarás las formas y salidas.